# 🎮 ValoCoach: Self-Attention Sequence Transformer for Valorant Match Dynamics
### Deep Learning on Round-by-Round Momentum, Economy & Tilt Inflection Points

This notebook trains the **ValoSequenceTransformer** on 37,642 complete 24-round match sequences (~755,000 rounds) extracted from 6 years of VCT professional play.

**Hardware requirement:** Free Kaggle / Google Colab Nvidia Tesla T4 GPU (Runtime: ~12-15 minutes).

In [ ]:
# 1. Environment Verification & Imports
import os, sys, time, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using compute device: {device}")
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"VRAM Available: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

In [ ]:
# 2. Load Extracted Sequence Dataset
data_path = "data/round_sequences.npz"
if not os.path.exists(data_path) and os.path.exists("round_sequences.npz"):
    data_path = "round_sequences.npz"

print(f"Loading {data_path}...")
archive = np.load(data_path)
X = archive["X"]          # [37642, 24, 14]
y_round = archive["y_round"] # [37642, 24]
y_match = archive["y_match"] # [37642]
mask = archive["mask"]    # [37642, 24]

print(f"Dataset loaded: {len(X):,} sequences with {mask.sum():,} active rounds.")

# Train (80%) / Val (10%) / Test (10%) Split
indices = np.random.RandomState(42).permutation(len(X))
n_train = int(len(X) * 0.8)
n_val = int(len(X) * 0.1)

train_idx = indices[:n_train]
val_idx = indices[n_train:n_train + n_val]
test_idx = indices[n_train + n_val:]

train_ds = TensorDataset(
    torch.from_numpy(X[train_idx]),
    torch.from_numpy(y_round[train_idx]),
    torch.from_numpy(y_match[train_idx]),
    torch.from_numpy(mask[train_idx])
)
val_ds = TensorDataset(
    torch.from_numpy(X[val_idx]),
    torch.from_numpy(y_round[val_idx]),
    torch.from_numpy(y_match[val_idx]),
    torch.from_numpy(mask[val_idx])
)
test_ds = TensorDataset(
    torch.from_numpy(X[test_idx]),
    torch.from_numpy(y_round[test_idx]),
    torch.from_numpy(y_match[test_idx]),
    torch.from_numpy(mask[test_idx])
)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=128, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=128, shuffle=False)

print(f"Batches: Train={len(train_loader)}, Val={len(val_loader)}, Test={len(test_loader)}")

In [ ]:
# 3. Define the ValoSequenceTransformer Architecture
class ValoSequenceTransformer(nn.Module):
    def __init__(self, input_dim=14, hidden_dim=128, num_heads=4, num_layers=3, dropout=0.1, max_rounds=24):
        super().__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.max_rounds = max_rounds

        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        self.pos_embedding = nn.Parameter(torch.randn(1, max_rounds, hidden_dim) * 0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim * 2,
            dropout=dropout,
            batch_first=True,
            activation="gelu"
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Multi-Task Heads
        self.round_win_head = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.GELU(),
            nn.Linear(64, 1)
        )
        self.reconstruct_head = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.GELU(),
            nn.Linear(64, input_dim)
        )
        self.match_win_head = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.GELU(),
            nn.Linear(64, 1)
        )
        self.archetype_head = nn.Linear(hidden_dim, 16)

    def forward(self, x, mask=None):
        B, T, D = x.shape
        h = self.input_proj(x) + self.pos_embedding[:, :T, :]
        padding_mask = ~mask if mask is not None else None
        encoded = self.transformer(h, src_key_padding_mask=padding_mask)

        round_win_logits = self.round_win_head(encoded).squeeze(-1)
        reconstructed = self.reconstruct_head(encoded)

        if mask is not None:
            mask_expanded = mask.unsqueeze(-1).float()
            pooled = (encoded * mask_expanded).sum(dim=1) / mask_expanded.sum(dim=1).clamp(min=1.0)
        else:
            pooled = encoded.mean(dim=1)

        match_win_logit = self.match_win_head(pooled).squeeze(-1)
        archetype_embedding = self.archetype_head(pooled)

        return round_win_logits, reconstructed, match_win_logit, archetype_embedding

model = ValoSequenceTransformer().to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"ValoSequenceTransformer initialized! Total Trainable Parameters: {total_params:,}")

In [ ]:
# 4. Training Loop with Mixed Precision & Cosine Scheduling
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
epochs = 15
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-5)
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

bce_loss = nn.BCEWithLogitsLoss(reduction="none")
mse_loss = nn.MSELoss(reduction="none")

print(f"Beginning {epochs}-Epoch Training Sequence...")
start_time = time.time()

for epoch in range(1, epochs + 1):
    model.train()
    total_train_loss = 0.0
    
    for bx, by_round, by_match, bmask in train_loader:
        bx, by_round, by_match, bmask = bx.to(device), by_round.to(device), by_match.to(device), bmask.to(device)
        optimizer.zero_grad()
        
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            round_logits, recon, match_logit, _ = model(bx, bmask)
            
            # 1. Round win loss (masked on active rounds)
            loss_round = (bce_loss(round_logits, by_round) * bmask.float()).sum() / bmask.float().sum().clamp(min=1.0)
            
            # 2. Match winner loss
            loss_match = bce_loss(match_logit, by_match).mean()
            
            # 3. Self-supervised reconstruction loss (latent anomaly detection)
            loss_recon = (mse_loss(recon, bx).mean(dim=-1) * bmask.float()).sum() / bmask.float().sum().clamp(min=1.0)
            
            # Multi-Task Joint Objective
            loss = loss_round + 0.5 * loss_match + 0.2 * loss_recon
            
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        
        total_train_loss += loss.item() * len(bx)
        
    scheduler.step()
    
    # Validation Phase
    model.eval()
    val_loss = 0.0
    correct_rounds = 0
    total_rounds = 0
    
    with torch.no_grad():
        for bx, by_round, by_match, bmask in val_loader:
            bx, by_round, by_match, bmask = bx.to(device), by_round.to(device), by_match.to(device), bmask.to(device)
            round_logits, recon, match_logit, _ = model(bx, bmask)
            
            loss_round = (bce_loss(round_logits, by_round) * bmask.float()).sum() / bmask.float().sum().clamp(min=1.0)
            loss_match = bce_loss(match_logit, by_match).mean()
            loss_recon = (mse_loss(recon, bx).mean(dim=-1) * bmask.float()).sum() / bmask.float().sum().clamp(min=1.0)
            v_loss = loss_round + 0.5 * loss_match + 0.2 * loss_recon
            val_loss += v_loss.item() * len(bx)
            
            preds = (torch.sigmoid(round_logits) > 0.5).float()
            correct_rounds += ((preds == by_round) * bmask.float()).sum().item()
            total_rounds += bmask.float().sum().item()
            
    val_acc = (correct_rounds / total_rounds) * 100 if total_rounds > 0 else 0
    print(f"Epoch {epoch:02d}/{epochs} | Train Loss: {total_train_loss/len(train_ds):.4f} | Val Loss: {val_loss/len(val_ds):.4f} | Round Win Accuracy: {val_acc:.1f}%")

print(f"Training sequence completed in {(time.time() - start_time) / 60:.2f} minutes!")

In [ ]:
# 5. Extract Hidden Patterns: Visualizing Momentum Trajectory & Tilt Inflection
import matplotlib.pyplot as plt

model.eval()
sample_x, sample_y, sample_match, sample_mask = test_ds[42]
sample_x_batch = sample_x.unsqueeze(0).to(device)
sample_mask_batch = sample_mask.unsqueeze(0).to(device)

with torch.no_grad():
    round_logits, recon, match_logit, archetype = model(sample_x_batch, sample_mask_batch)
    win_probs = torch.sigmoid(round_logits).squeeze(0).cpu().numpy()
    recon_errors = (F.mse_loss(recon.squeeze(0), sample_x.to(device), reduction="none").mean(dim=-1) * sample_mask.to(device).float()).cpu().numpy()

valid_len = int(sample_mask.sum().item())
rounds_range = range(1, valid_len + 1)

plt.figure(figsize=(12, 4))
plt.plot(rounds_range, win_probs[:valid_len], marker="o", color="#ff4655", label="Predicted Win Probability")
plt.plot(rounds_range, recon_errors[:valid_len], marker="s", color="#f59e0b", label="Reconstruction Anomaly (Tilt/Irregularity)")
plt.axhline(0.5, color="gray", linestyle="--", alpha=0.5)
plt.title("ValoCoach Neural Sequence Trajectory: Round Win Expectancy & Tilt Spikes")
plt.xlabel("Round Number")
plt.ylabel("Metric Value")
plt.grid(True, alpha=0.2)
plt.legend()
plt.show()

tilt_round = int(np.argmax(recon_errors[:valid_len])) + 1
print(f"🎯 Detected Tactical Inflection Point: Round {tilt_round} exhibited the highest behavioral divergence!")

In [ ]:
# 6. Export to ONNX for Production Inference
model.eval()
dummy_x = torch.randn(1, 24, 14, dtype=torch.float32).to(device)
dummy_mask = torch.ones(1, 24, dtype=torch.bool).to(device)

onnx_path = "data/valo_transformer.onnx"
os.makedirs(os.path.dirname(onnx_path), exist_ok=True)

print(f"Exporting PyTorch model to ONNX: {onnx_path}...")
torch.onnx.export(
    model,
    (dummy_x, dummy_mask),
    onnx_path,
    input_names=["round_sequence", "round_mask"],
    output_names=["round_win_logits", "reconstructed", "match_win_logit", "archetype_embedding"],
    dynamic_axes={
        "round_sequence": {0: "batch_size"},
        "round_mask": {0: "batch_size"},
        "round_win_logits": {0: "batch_size"},
        "reconstructed": {0: "batch_size"},
        "match_win_logit": {0: "batch_size"},
        "archetype_embedding": {0: "batch_size"}
    },
    opset_version=14
)

print("✓ ONNX Export complete!")
print(f"File Size: {os.path.getsize(onnx_path) / (1024 * 1024):.2f} MB")
print("
You can now drop this file directly into the ml/data/ directory of your ValoCoach repository!")